**Prerequisites:**

Foundational knowledge of NLP, Spacy, theoretical background of Encoder- Decoder Networks and end-to-end ML models.

**Objective:**

1. To understand the implementation of encoder- decoder networks using pytorch in a high- level and understand the workflow.
2. To understand the implementation of encoder- decoder networks in detail and understand all the tools, techniques and methodologies required to understand the implementation.



**torchtext:**

The torchtext package consists of data processing utilities and popular datasets for natural language. It's important to note that some import errors might be caused if appropriate version of torchtext is not imported in case of libraries like 'Field'. Version 0.8.1 is appropriate and adequate for our requirements here.

In [ ]:
!pip install -U torchtext==0.8.1

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


Libraries to be used:

**[torch:](https://pytorch.org/docs/stable/torch.html )**The torch package contains data structures for multi-dimensional tensors and defines mathematical operations over these tensors. Additionally, it provides many utilities for efficient serialization of Tensors and arbitrary types, and other useful utilities.


**[torch.nn:](https://pytorch.org/docs/stable/nn.html)**These are the basic building blocks for graphs of different kinds.


**[torch.optim:](https://pytorch.org/docs/stable/optim.html )** is a package implementing various optimization algorithms. Most commonly used methods are already supported, and the interface is general enough, so that more sophisticated ones can also be easily integrated in the future.


**torchtext:** It used below has been defined earlier.

**torchtext.data**
The data module provides the following:

1. Ability to define a preprocessing pipeline
2. Batching, padding, and numericalizing (including building a vocabulary object)
3. Wrapper for dataset splits (train, validation, test)
4. Loader a custom NLP dataset

For more information: [Click Here](https://torchtext.readthedocs.io/en/latest/data.html)

In [ ]:
# Import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchtext.datasets import Multi30k #German to English dataset
from torchtext.data import Field, BucketIterator
import numpy as np
import spacy
import random
from torch.utils.tensorboard import SummaryWriter  # to print to tensorboard
import torch
import spacy
from torchtext.data.metrics import bleu_score
import sys

**[Field:](https://torchtext.readthedocs.io/en/latest/data.html#fields )**

Defines a datatype together with instructions for converting to Tensor.  Field class models common text processing datatypes that can be represented by tensors. It holds a Vocab object that defines the set of possible values for elements of the field and their corresponding numerical representations. The Field object also holds other parameters relating to how a datatype should be numericalized, such as a tokenization method and the kind of Tensor that should be produced.  If a Field is shared between two columns in a dataset (e.g., question and answer in a QA dataset), then they will have a shared vocabulary.

**[Bucket Iterator:](https://torchtext.readthedocs.io/en/latest/data.html#bucketiterator)**

Defines an iterator that batches examples of similar lengths together.  Minimizes amount of padding needed while producing freshly shuffled batches for each new epoch.


**Multi30K:**

Multi30K is an extension of the Flickr30K dataset (Young et al., 2014) with 31014 German translations of English descriptions and 155,070 independently collected German descriptions.

**[SummaryWriter:](https://pytorch.org/docs/stable/tensorboard.html)**

The SummaryWriter class provides a high-level API to create an event file in a given directory and add summaries and events to it.

**[spaCy:](https://spacy.io/)**

spaCy is a free open-source library for Natural Language Processing in Python. It features NER, POS tagging, dependency parsing, word vectors and more.


For German Tokenizer:

'de' is the nomenclature for the spacy to be downloaded in and for german language.

For English Tokenizer:

'en' is the nomenclature for the spacy to be downloaded in and for german language.


In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
!python -m spacy download de

2023-03-30 07:33:20.970396: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2023-03-30 07:33:20.970507: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2023-03-30 07:33:20.970527: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nvidia GPU with TensorRT, please make sure the missing libraries mentioned above are installed properly.
⚠ As of spaCy v3.0, shortcuts like 'de' are deprecated. Please use the
full pipeline package name 'de_core_news_sm' instead.
Looking in indexes: https://pypi.org/simpl

In [ ]:
spacy_ger = spacy.load("de_core_news_sm")


In [ ]:
!python -m spacy download en

2023-03-30 07:33:40.684058: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2023-03-30 07:33:40.684203: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2023-03-30 07:33:40.684225: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nvidia GPU with TensorRT, please make sure the missing libraries mentioned above are installed properly.
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
Looking in indexes: https://pypi.org/simple

In [ ]:
spacy_eng = spacy.load("en_core_web_sm")

In [ ]:
# Tokenization of German Language
def tokenize_ger(text):
    return [tok.text for tok in spacy_ger.tokenizer(text)]

In [ ]:
# Tokenization of English Language

def tokenize_eng(text):
    return [tok.text for tok in spacy_eng.tokenizer(text)]

## Preprocessing of Text

In [ ]:
# Applyling Tokenization , lowercase and special Tokens for preprocessing
german = Field(tokenize = tokenize_ger,lower = True,init_token = '<sos>',eos_token = '<eos>')

/usr/local/lib/python3.9/dist-packages/torchtext/data/field.py:150: UserWarning: Field class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)


**Token description:**

initialization token (init_token) description: 'sos': start of sentence;
termination token (eos_token) description: :'eos': end of sentence

'sos' and 'eos' tokens are used to mark the boundarious of vectored information, respectively the starting and the ending boundaries.

In [ ]:
english = Field(tokenize = tokenize_eng,lower = True,init_token = '<sos>',eos_token = '<eos>')

**Dataset split:**

Divide the Multi30k dataset data into the training, validation and testing sets.

In [ ]:
# Dwonloading Dataset and storing them
train_data, valid_data, test_data = Multi30k.splits(
    exts=(".de", ".en"), fields=(german, english)
)

downloading training.tar.gz


training.tar.gz: 100%|██████████| 1.21M/1.21M [00:03<00:00, 386kB/s]


downloading validation.tar.gz


validation.tar.gz: 100%|██████████| 46.3k/46.3k [00:00<00:00, 113kB/s] 


downloading mmt_task1_test2016.tar.gz


mmt_task1_test2016.tar.gz: 100%|██████████| 66.2k/66.2k [00:00<00:00, 106kB/s]
/usr/local/lib/python3.9/dist-packages/torchtext/data/example.py:78: UserWarning: Example class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('Example class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.', UserWarning)


**Vocabulary:**

Create a vocabulary with a size big enough for the purpose of the project. In our case, we've chosen that size to be 10000.

In [ ]:
# Creating vocabulary in each language
german.build_vocab(train_data,max_size = 10000,min_freq = 2)
english.build_vocab(train_data,max_size = 10000,min_freq = 2)


Libraries required for Encoder and Decoder:

**1. Dropout**

During training, randomly removes (zeroes) some of the elements of the input tensor with probability p using samples (from a Bernoulli distribution).
Dropout improves time- efficiency while risking the performance metrics like accuracy with loss of information, so 'Dropout' should be properly tweaked to do the proper trade- off of accuracy and efficiency.

**2. Embedding**

Embedding module is used to create multi- dimensioned embedding versions of the elements of the given vector in a customized way over given vocabulary_size and required number of dimensions.

**3. LSTM:**

This module applies a multi-layer long short-term memory (LSTM) RNN to an input sequence.

**4. Linear**

This module applies a linear transformation to the incoming data

**5. Module:**

Base class for all neural network modules.


In [ ]:
from torch.nn import Dropout
from torch.nn import Embedding
from torch.nn import LSTM
from torch.nn import Linear
from torch.nn import Module




```
# This is formatted as code
```



**Encoder Class Building Steps:**

1. '__init__()' method:

Initializes and gets fed in the parameters: input_size, embedding_size, hidden_size, num_layers, p (probability for the Dropout).

__init__ method also prepares the 'self.embedding' and 'self.rnn' attributes using the Embedding and the LSTM modules respectively.

2. 'forward()' method:

The 'forward()' methods creates an 'embedding' attribute by applying dropout on the 'self.embedding' attribute generated earlier. Also, by applying 'embedding'to the LSTM module, we generate 1. outputs 2. tuple of 'hidden' and 'cell'.

In [ ]:
# Defining the Encoder part of the model
class Encoder(Module):

    def __init__(self, input_size, embedding_size, hidden_size, num_layers, p):
        super(Encoder, self).__init__()
        self.dropout = Dropout(p)
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = Embedding(input_size, embedding_size)
        self.rnn = LSTM(embedding_size, hidden_size, num_layers, dropout=p)

    def forward(self, x):
        # x shape: (seq_length, N) where N is batch size

        embedding = self.dropout(self.embedding(x))
        # embedding shape: (seq_length, N, embedding_size)

        outputs, (hidden, cell) = self.rnn(embedding)
        # outputs shape: (seq_length, N, hidden_size)

        return hidden, cell

**Decoder Class Building Steps:**

1. '__init__()' method:

Initializes and gets fed in the parameters: input_size, embedding_size, hidden_size, output_size, num_layers, p (probability for the Dropout). (change from encoder: output_size).

__init__ method also prepares the 'self.embedding', 'self.rnn' and 'self.fc' attributes using the Embedding, the LSTM and the Linear modules respectively.

2. 'forward()' method:

The 'forward()' methods creates an 'embedding' attribute by applying dropout on the 'self.embedding' attribute generated earlier. Also, by applying 'embedding'to the LSTM module, we generate 1. outputs 2. tuple of 'hidden' and 'cell'.

Also, predictions are obtained from applying outputs to self.fc. And, then, they are squeezed using squeeze(0) command.

Eventually, predictions, hidden and cell are returned by the forward() method.

In [ ]:
# Defining the Decoder part

class Decoder(Module):
    def __init__(
        self, input_size, embedding_size, hidden_size, output_size, num_layers, p):
        super(Decoder, self).__init__()
        self.dropout = Dropout(p)
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = Embedding(input_size, embedding_size)
        self.rnn = LSTM(embedding_size, hidden_size, num_layers, dropout=p)
        self.fc = Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell):
        # x shape: (N) where N is for batch size, we want it to be (1, N), seq_length
        # is 1 here because we are sending in a single word and not a sentence
        x = x.unsqueeze(0)

        embedding = self.dropout(self.embedding(x))
        # embedding shape: (1, N, embedding_size)

        outputs, (hidden, cell) = self.rnn(embedding, (hidden, cell))
        # outputs shape: (1, N, hidden_size)

        predictions = self.fc(outputs)

        # predictions shape: (1, N, length_target_vocabulary) to send it to
        # loss function we want it to be (N, length_target_vocabulary) so we're
        # just gonna remove the first dim
        predictions = predictions.squeeze(0)

        return predictions, hidden, cell

**Seq2Seq Class Building Steps:**

1. '__init__()' method:

Initializes the class along with respective assignment of 'encoder' and 'decoder' variable values to the self.encoder and self.decoder variables.


2. 'forward()' method:

Given the source and target information and teacher_force_ratio, we initiate the proceedings.

From target_len, batch_size and target_vocab_size, we get the outputs. And, by applying self.encoder() on the source, 'hidden' and 'cell' variable values are obtained.

Applying the first input to the decoder (i.e, 'sos' token), hidden and cell variable values to the self.decoder(), we obtain the triple- variable output: outputs, hidden and cell.

We then store the next output prediction.

Then, we get the best word the Decoder predicted (index in the vocabulary).

With probability of teacher_force_ratio,we take the actual next word         otherwise we take the word that the Decoder predicted it to be. Teacher Forcing is used so that the model gets used to seeing similar inputs at training and testing time, if teacher forcing is 1, then inputs at test time might be completely different than what the network is used to.

In [ ]:
# Defining the complete model
class Seq2Seq(Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_force_ratio=0.5):
        batch_size = source.shape[1]
        target_len = target.shape[0]
        target_vocab_size = len(english.vocab)

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(device)

        hidden, cell = self.encoder(source)

        # Grab the first input to the Decoder which will be <SOS> token
        x = target[0]

        for t in range(1, target_len):
            # Use previous hidden, cell as context from encoder at start
            output, hidden, cell = self.decoder(x, hidden, cell)

            # Store next output prediction
            outputs[t] = output

            # Get the best word the Decoder predicted (index in the vocabulary)
            best_guess = output.argmax(1)

            x = target[t] if random.random() < teacher_force_ratio else best_guess

        return outputs

Tweak the hyperparamters (majorly more determining parameters such as num_epochs, learning_rate and batch_size) and observe the final performance metrics (in our case, bleu score).
And, choose the variations of values that get the optimum results.

I have chosen the following values as the optimum hyperparameter values after extensive experimentation.

In [ ]:
# Hyperparameters
num_epochs = 100
learning_rate = 0.001
batch_size = 256


Model hyperparameters also have to be set.

Appropriate device has to be chosen. (Effective strategy is that 'Choose the best device available')

The input size for the encoder and decoder have to be chosen in the case of our MT project with respect to respective vocabulary sizes/

The other parameters to also be set: encoder_embedding_size, decoder_embedding_size, hidden_size, num_layers, enc_dropout and dec_dropout.

In [ ]:
# Model hyperparameters
load_model = False
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
input_size_encoder = len(german.vocab)
input_size_decoder = len(english.vocab)
output_size = len(english.vocab)
encoder_embedding_size = 300
decoder_embedding_size = 300

hidden_size = 1024
num_layers = 2
enc_dropout = 0.5
dec_dropout = 0.5


Use the SummaryWriter() command to get the loss plot.


In [ ]:
writer = SummaryWriter(f'runs/Loss_plot')
step = 0

Use the BucketIterator to create train_iterator, validation_iterator and test_iterator from train_data, valid_data, test_data.

Also, here we set the appropriate values for the parameters: batch_size, sort_within_batch (boolean parameter),sort_key and device.

In [ ]:
train_iterator, validation_iterator, test_iterator = BucketIterator.splits(
    (train_data, valid_data, test_data),
     batch_size = batch_size,
     sort_within_batch = True,
     sort_key = lambda x:len(x.src),
     device = device)

/usr/local/lib/python3.9/dist-packages/torchtext/data/iterator.py:48: UserWarning: BucketIterator class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)



1. Create the encoder_net object:

Use the Encoder class. Feed the following parameters to the class: input_size_encoder, encoder_embedding_size, hidden_size,num_layers,
enc_dropout, device.

2. Create the decoder_net object:

Use the Decoder class. Feed the following parameters to the class:
input_size_decoder, decoder_embedding_size, hidden_size,output_size,
num_layers, dec_dropout, device

In [ ]:
encoder_net = Encoder(input_size_encoder,
                      encoder_embedding_size,
                      hidden_size,num_layers,
                      enc_dropout).to(device)


decoder_net = Decoder(input_size_decoder,
                      decoder_embedding_size,
                      hidden_size,output_size,num_layers,
                      dec_dropout).to(device)

Create the model using the encoder_net and the decoder_net.

Then, model has been chosen to be optimized using adam optimizer. ALso, during optimization, we set the appropriate learning_rate.

In [ ]:
model = Seq2Seq(encoder_net, decoder_net).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Import the following classes from the from torch.nn library: Dropout, Embedding, LSTM, Linear, Module, CrossEntropyLoss.

In [ ]:
from torch.nn import Dropout
from torch.nn import Embedding
from torch.nn import LSTM
from torch.nn import Linear
from torch.nn import Module
from torch.nn import CrossEntropyLoss

In [ ]:
pad_idx = english.vocab.stoi['<pad>']
criterion = CrossEntropyLoss(ignore_index = pad_idx)

Perform the major operations of machine- translation through translate_sentence() method.

The inputs to the method are: model, sentence, german, english, device, max_length.

Steps:

1. Create tokens using spacy and everything in lower case (which is what our vocab is).

2. Add 'SOS' and 'EOS' tokens in beginning and end respectively.

3. Go through each german token and convert the token to an index and obtain text_to_indices.

4. Using the text_to_indices, obtain the sentence_tensor.

5.  Obtain the hidden, cell state values by applying model.encoder() to the sentence_tensor.

6. Initialize the 'outputs' variable with the 'sos' token.

7. Begin looping the operations with the counter ranging from 0 to the max_length.

  7.1. By applying [outputs[-1]] to torch.LongTensor(), we obtain previous_word.

  7.2. By applying previous_word, hidden, cell inputs to the model.decoder(), we obtain the triplet of output, hidden and cell.

  7.3. Obtain the best_guess by applying argmax(1) to the output.

  7.4. Append the best_guess values iteratively to the 'outputs' variable.

  7.5.Break condition:  if the end of the sentence is reached (signified by 'eos')

8. For the respective index in outputs. apply itos(integer_to_string); The accumulated sentence signifies the translated_sentence.

9. Remove the start token. Starting from index 1 and onwards represents the cleaned up version of the translated_sentence.








In [ ]:
def translate_sentence(model, sentence, german, english, device, max_length=50):

    if type(sentence) == str:
        tokens = [token.text.lower() for token in spacy_ger(sentence)]
    else:
        tokens = [token.lower() for token in sentence]

    # print(tokens)

    # sys.exit()
    # Add <SOS> and <EOS> in beginning and end respectively
    tokens.insert(0, german.init_token)
    tokens.append(german.eos_token)

    # Go through each german token and convert to an index
    text_to_indices = [german.vocab.stoi[token] for token in tokens]

    # Convert to Tensor
    sentence_tensor = torch.LongTensor(text_to_indices).unsqueeze(1).to(device)

    # Obtain the hidden, cell state values by applying model.encoder() to the sentence_tensor
    with torch.no_grad():
        hidden, cell = model.encoder(sentence_tensor)

    outputs = [english.vocab.stoi["<sos>"]]

    for _ in range(max_length):
        previous_word = torch.LongTensor([outputs[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(previous_word, hidden, cell)
            best_guess = output.argmax(1).item()

        outputs.append(best_guess)

        # Model predicts it's the end of the sentence
        if output.argmax(1).item() == english.vocab.stoi["<eos>"]:
            break

    translated_sentence = [english.vocab.itos[idx] for idx in outputs]

    # remove start token
    return translated_sentence[1:]

Give the inputs: state and filename (in our case: "my_checkpoint.pth.tar") to the save_checkpoint() method. The state and the filename are saved using the torch,save() method.

In [ ]:
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)


In [ ]:
def load_checkpoint(checkpoint, model, optimizer):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

If the variable 'load_model' is True, we load the checkpoint using method load_checkpoint() with torch.load("my_checkpoint.pth.tar"), model, optimizer as the parameters for the load_checkpoint() method.

In [ ]:
if load_model:
    load_checkpoint(torch.load("my_checkpoint.pth.tar"), model, optimizer)


Choose an example sentence in the german language.

In [ ]:
sentence = "Ein brauner und ein schwarzer Hund laufen"


Perform iteratively the following operations with the iteration happening in the range of 0 to the assigned number of epochs.

1. Save the checkpoint obtained from the dictionary operations: {"state_dict": model.state_dict(), "optimizer": optimizer.state_dict()

2. Evaluate the model using model.eval().

3. Apply the parameters: model, sentence, german, english, device, max_length to the method translate_sentence() to obtain the translated_sentence.

4. Apply the model.train() method to begin the model training process.

5. Begin a loop for the enumerated version of train_iterator with the iterating variables being batch_idx, batch.

5.1.Get input and targets and get to the device.
    
5.2. Fprward propagation: Apply inp_data, target to the model() method to obtain the output.


5.3. Output is of shape (trg_len, batch_size, output_dim) but Cross Entropy Loss
doesn't take input in that form. For example if we have MNIST we want to have
output to be: (N, 10) and targets just (N). Here we can view it in a similar
way that we have output_words * batch_size that we want to send in into
our cost function, so we need to do some reshapin. While we're at it

5.4. Remove the start tokens from the output and the target and we also apply reshaping subsequently.

5.5. Apply optimizer.zero_grad() method.

5.6. Apply output, target as inputs to the criterion() method to obtain the loss.

5.7. Back propagation: Apply the method: loss.backward()

5.8. Clip to avoid exploding gradient issues, makes sure grads are within a healthy range

5.9. Gradient descent step: Apply optimizer.step()

5.10. Plot to tensorboard using writer.add_scalar() method with "Training loss", loss, global_step as the parameters.

5.11. Increment the step variable by 1.
    

In [ ]:
for epoch in range(num_epochs):
    print(f"[Epoch {epoch} / {num_epochs}]")

    checkpoint = {"state_dict": model.state_dict(), "optimizer": optimizer.state_dict()}
    save_checkpoint(checkpoint)

    model.eval()

    translated_sentence = translate_sentence(
        model, sentence, german, english, device, max_length=50
    )

    print(f"Translated example sentence: \n {translated_sentence}")

    model.train()

    for batch_idx, batch in enumerate(train_iterator):
        inp_data = batch.src.to(device)
        target = batch.trg.to(device)

        # Forward prop
        output = model(inp_data, target)

        output = output[1:].reshape(-1, output.shape[2])
        target = target[1:].reshape(-1)

        optimizer.zero_grad()
        loss = criterion(output, target)

        # Back prop
        loss.backward()

        # Clip to avoid exploding gradient issues, makes sure grads are
        # within a healthy range
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)

        # Gradient descent step
        optimizer.step()

        # Plot to tensorboard
        writer.add_scalar("Training loss", loss, global_step=step)
        step += 1

[Epoch 0 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['walkway', 'string', 'string', 'storefronts', 'sand', 'sand', 'warmers', 'teenagers', 'spinner', 'mothers', 'lighting', 'poorly', 'storefronts', 'lighting', 'comforts', 'sale', 'comforts', 'comforts', 'aerial', 'aerial', 'fat', 'fat', 'handicapped', 'asian', 'fat', 'sunshine', 'spinner', 'spinner', 'poorly', 'classroom', 'classroom', 'hamburgers', 'hamburgers', 'hamburgers', 'assisted', 'list', 'string', 'string', 'whom', 'hamburgers', 'hamburgers', 'hamburgers', 'sand', 'hamburgers', 'string', 'string', 'sand', 'referees', 'brace', 'warmers']


/usr/local/lib/python3.9/dist-packages/torchtext/data/batch.py:23: UserWarning: Batch class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)


[Epoch 1 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['a', 'man', 'in', 'a', 'a', 'a', 'a', '.', '<eos>']
[Epoch 2 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['a', 'dog', 'is', 'a', 'a', 'a', 'a', 'a', '.', '<eos>']
[Epoch 3 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['a', 'black', 'dog', 'is', 'running', 'in', 'a', 'of', '.', '<eos>']
[Epoch 4 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['a', 'black', 'and', 'white', 'dog', 'and', 'a', 'dog', '.', '<eos>']
[Epoch 5 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['a', 'black', 'and', 'white', 'dog', 'running', 'through', 'the', 'grass', '.', '<eos>']
[Epoch 6 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['a', 'brown', 'and', 'brown', 'dog', 'is', 'running', 'through', '.', '<eos>']
[Epoch 7 / 80]
=> Saving checkpoint
Translated example sentence: 
 ['a', 'brown', 'dog', 'and', 'a', 'brown', 'dog', '.', '<eos>']
[Epoch 8 / 80]
=> Saving checkpoint
Translate

1. Define the method for BLEU score computation: bleu() ( parameters: data, model, german, english, device).

2. Initialize the lists for the targets and the outputs variables.

3. Iterate with the terating variable 'example' over 'data'.

  3.1. Get the src and trg from the example.

  3.2. Make a prediction using the method translate_sentence() with the following parameters given: model, src, german, english, device.

  3.3. Remove 'EOS' token from the obtained prediction.

  3.4. Append the trg values to the targets variable and prediction values to the outputs variable.

4. Apply the bleu_score() method on the outputs and targets.

In [ ]:
def bleu(data, model, german, english, device):
    targets = []
    outputs = []

    for example in data:
        src = vars(example)["src"]
        trg = vars(example)["trg"]

        prediction = translate_sentence(model, src, german, english, device)
        prediction = prediction[:-1]  # remove <eos> token

        targets.append([trg])
        outputs.append(prediction)

    return bleu_score(outputs, targets)


Obtain the bleu_score from the first 100 test_data with the following as the parameters: model, german, english, device.


In [ ]:

score = bleu(test_data[1:100], model, german, english, device)
print(f"Bleu score {score*100:.2f}")

Bleu score 18.48
